# 4. Conversation memory and multi-query retrieval

> **Course concepts used here**
>
| Course notebook | What we use it for |
|---|---|
| `L_LLM_APIs/4_conversation_history` | sending previous turns with each request |
| `N_LangChain/04_langchain-memory` | window memory (`ConversationBufferWindowMemory`) |
| `N_LangChain/11_langchain-multi-query` | `MultiQueryRetriever`: several rewrites of the question |

In [1]:
import os, sys, warnings
from pathlib import Path

# Run from the repo root so `ironstore_assistant` can be imported
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from ironstore_assistant import config
from ironstore_assistant.fake_llm import FakeLLM
from ironstore_assistant.rag import get_llm

HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))
print("LLM:", config.CHAT_MODEL if HAS_KEY else "scripted FakeLLM (no OPENAI_API_KEY found)")

LLM: scripted FakeLLM (no OPENAI_API_KEY found)


In [3]:
from ironstore_assistant.build_index import ensure_index
from ironstore_assistant.index import load_index
from ironstore_assistant.rag import Assistant, Retriever, load_products

ensure_index()
retriever = Retriever(load_index(), load_products())

## 4.1 Memory: the problem

*"And how often do I have to change it?"* means nothing to a vector search: "it" is the password from the previous
turn. We keep a **window** of the last turns (like `ConversationBufferWindowMemory`) and use it in two places:

1. to **condense** the follow-up into a standalone question for retrieval,
2. in the answer prompt, so the reply stays consistent with the conversation.

In [4]:
from ironstore_assistant.rag import to_messages

history = [("user", "What is the minimum password length?"),
           ("assistant", "Passwords must be at least 14 characters long [1]."),
           ("user", "Thanks"), ("assistant", "You're welcome!")]
to_messages(history, window=1)  # only the last turn is kept

[HumanMessage(content='Thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content="You're welcome!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [5]:
from ironstore_assistant.prompts import CONDENSE_PROMPT

llm = get_llm() if HAS_KEY else FakeLLM(text="How often must IronStore passwords be changed?")
assistant = Assistant(retriever.store, llm, retriever.products, multi_query=False)

follow_up = "And how often do I have to change it?"
standalone = assistant.condense(follow_up, to_messages(history[:2]))
print("standalone question:", standalone)

standalone question: How often must IronStore passwords be changed?


In [6]:
def top_docs(queries, k=4):
    return [f'{d.metadata["title"]} p.{d.metadata["page"]}' for d in retriever.search(queries, k=k)]

print("raw follow-up  ->", top_docs([follow_up]))
print("condensed      ->", top_docs([standalone]))

raw follow-up  -> ['Password & Access Management Policy p.1', 'Password & Access Management Policy p.2', 'Customer Support Knowledge Base Style Guide p.2', 'Password & Access Management Policy p.1']
condensed      -> ['Password & Access Management Policy p.2', 'Password & Access Management Policy p.3', 'Password & Access Management Policy p.4', 'Password & Access Management Policy p.2']


## 4.2 Multi-query: the problem

Employees and policies don't use the same words. *"Time off"* never appears in the Leave & Absence Policy, which
talks about *"annual leave"*. With weaker embeddings that question misses the policy:

In [7]:
question = "How do I request time off?"
top_docs([question], k=5)

['IronStore Product Catalog — Skincare p.31',
 'IronStore Product Catalog — Health & Wellness p.31',
 'Customer Support Knowledge Base Style Guide p.4',
 'Customer Support Knowledge Base Style Guide p.4',
 'Customer Support Knowledge Base Style Guide p.2']

The fix from `N_LangChain/11`: ask the LLM for a few rewrites in "policy vocabulary", search with each, and merge
the results.

In [8]:
from ironstore_assistant.prompts import MULTI_QUERY_PROMPT

if HAS_KEY:
    queries = Assistant(retriever.store, get_llm(), multi_query=True).expand(question)
else:  # what a model typically writes for this prompt
    queries = [question, "annual leave request procedure", "how to book holiday leave in the HR system",
               "leave approval by line manager"]
queries

['How do I request time off?',
 'annual leave request procedure',
 'how to book holiday leave in the HR system',
 'leave approval by line manager']

In [9]:
top_docs(queries, k=5)

['Leave & Absence Policy p.4',
 'Leave & Absence Policy p.2',
 'Leave & Absence Policy p.4',
 'Leave & Absence Policy p.5',
 'Leave & Absence Policy p.2']

## 4.3 Merging the results: reciprocal rank fusion

`MultiQueryRetriever` simply takes the union of the results. We also **rank** them: a chunk gets `1 / (60 + rank)`
points from every query that found it, so chunks found by several rewrites float to the top.

In [10]:
toy = {"query A": ["leave p.2", "handbook p.3", "remote p.2"],
       "query B": ["leave p.2", "leave p.5", "handbook p.3"],
       "query C": ["leave p.5", "leave p.2", "payroll p.1"]}
scores = {}
for q, ranking in toy.items():
    for rank, doc in enumerate(ranking):
        scores[doc] = scores.get(doc, 0) + 1 / (60 + rank)
pd.Series(scores).sort_values(ascending=False).round(4)

leave p.2       0.0497
leave p.5       0.0331
handbook p.3    0.0325
remote p.2      0.0161
payroll p.1     0.0161
dtype: float64

## 4.4 Both together

In [11]:
if not HAS_KEY:
    from ironstore_assistant.rag import StructuredAnswer
    llm = FakeLLM(text="annual leave request procedure",
                  structured=StructuredAnswer(answer="Request leave in advance through PeopleHub [1]; it is "
                                              "confirmed once your line manager approves it [1].",
                                              citations=[1], confidence="high", answerable=True))
assistant = Assistant(retriever.store, llm, retriever.products, multi_query=True)
resp = assistant.ask("How do I request time off?")
print(resp.answer)
print("searched with:", resp.queries)
print("sources:", [s.label() for s in resp.sources])

Request leave in advance through PeopleHub [1]; it is confirmed once your line manager approves it [1].
searched with: ['How do I request time off?', 'annual leave request procedure']
sources: ['Leave & Absence Policy (p. 4)']


Next: letting the model choose its own tools → **notebook 5**.